# Dimensions in a sketch

`sketch.Measure(id)` measures the element or the point with that id, drawn or construction, and with `to=` it
measures between two of them: lengths, radii, angles and distances. A drawing built from references, rather than from
numbers, can be checked this way against its design intent.

In [1]:
import math
import planex as px

In [2]:
from docs_figures import Row, Vertices

## The bracket and the link

The example checks two sketches. The first is the bracket from [Parallel, perpendicular,
angle](../sketch/parallel-perpendicular-angle.md): its ramp was drawn at β = 35° to the ground and ends on the wall, a
= 10 away, so the back is a · tan β high (Figure 1a, with the ramp and the back highlighted). The second is the link
from [Tangents](../sketch/tangent.md), two bosses of radius r1 = 1.5 and r2 = 1 with their centers d = 6 apart. With
sin α = (r1 − r2) / d, the end arc of the large boss spans 180° + 2α, and the tangent lines meet at 2α and come
closest at the small boss, 2 r2 cos α apart (Figure 1b).

In [3]:
a, beta, t = 10.0, 35.0, 0.8
b = px.QuickSketch()
b.Move((-1, 0)).Line((a + 2, 0), id="ground", construction=True)
b.Move((a, -1)).Line((a, 8), id="wall", construction=True)

b.Move((0, 0)).Line("wall", angle=beta, id="ramp")
b.Line("ground", congruent="wall", id="back")
b.Close(id="base")

b.Move((0, 0), dir=(1, 0)).Line(14, parallel="ramp", distance=-t, id="ramp in", construction=True)
b.Move((0, 0), dir=(1, 0)).Line(a + 2, parallel="ground", distance=t, id="ground in", construction=True)
b.Move((0, 0), dir=(0, 1)).Line(9, parallel="wall", distance=t, id="wall in", construction=True)
b.Line("ground in", "wall in", congruent="ramp in")
b.Line("ground in", congruent="wall in")
b.Close()

link = px.QuickSketch()
link.Circle(0, 0, 1.5, id="boss1", construction=True)
link.Circle(6, 0, 1.0, id="boss2", construction=True)
link.Line(tangent=("boss1", "boss2"), id="upper")
link.Line(tangent=("boss1", "boss2"), reverse=True, id="lower")
link.Arc("lower", "upper", congruent="boss2", id="end2")
link.Arc("upper", "lower", congruent="boss1", id="end1")
link.Circle(0, 0, 0.6, id="hole1").Circle(6, 0, 0.4, id="hole2")
r1, r2, d = 1.5, 1.0, 6.0
alpha = math.degrees(math.asin((r1 - r2) / d))

In [4]:
Row((b, "The ramp and the back", {"fill": False, "highlight": [e for n in ("ramp", "back") for e in b.Edge(n).edges]}),
    (link, "The end arc and the tangents", {"fill": False, "highlight": [e for n in ("end1", "upper", "lower") for e in link.Edge(n).edges]}),
    caption="A bracket and a link, measured by their ids")

In [5]:
print("ramp to ground", b.Measure("ramp", to="ground")["angle"], "exact", beta)
print("back", b.Measure("back")["length"], "exact", a * math.tan(math.radians(beta)))

end1 = link.Measure("end1")
print(end1["type"], end1["radius"], end1["angle"], "exact", 180 + 2 * alpha)
between = link.Measure("upper", to="lower")
print("tangents meet at", between["angle"], "exact", 2 * alpha)
print("closest", between["distance"], "exact", 2 * r2 * math.cos(math.radians(alpha)))

ramp to ground 35.000000000000014 exact 35.0
back 7.0020753820971 exact 7.002075382097097
arc 1.5 189.56038369439833 exact 189.56038369439833
tangents meet at 9.560383694398318 exact 9.560383694398318
closest 1.9930434571835662 exact 1.9930434571835662


The distance between the holes is the center distance minus both radii:

In [6]:
print(link.Measure("hole1", to="hole2")["distance"], "exact", d - 0.6 - 0.4)

5.0 exact 5.0
